In [53]:
import pandas as pd
import polars as pl
from datetime import datetime, timedelta
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_halving_search_cv
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score,roc_curve, auc,mean_squared_error,recall_score,
precision_recall_curve,log_loss,roc_auc_score,average_precision_score,f1_score)
from sklearn.model_selection import train_test_split,RepeatedKFold,cross_val_score,TimeSeriesSplit,GridSearchCV,HalvingGridSearchCV
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor,MLPClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from tempfile import mkdtemp
from skorch import NeuralNetClassifier
from skorch.callbacks import EarlyStopping
from skorch.dataset import ValidSplit
from joblib import parallel_backend
import torch
import torch.nn as nn

In [54]:
feature_PATH = "../data/processed/features/feature_cols.csv"
feature_cols = pd.read_csv(feature_PATH)["feature"].tolist()

response_PATH = "../data/processed/response_cols/response_cols.csv"
response_cols = pd.read_csv(response_PATH)["response"].tolist()

train_data = pd.DataFrame() 
train_PATH = f"../data/processed/train/train_data.parquet"
train_data = pd.concat([train_data, pd.read_parquet(train_PATH)])

test_data = pd.DataFrame() 
test_PATH = f"../data/processed/test/test_data.parquet"
test_data = pd.concat([test_data , pd.read_parquet(test_PATH)])

In [55]:
common = train_data.columns.intersection(test_data.columns)
object_cols = train_data[common].select_dtypes(include=["object"]).columns

for col in object_cols:
    cats = pd.concat([train_data[col], test_data[col]]).dropna().unique()
    dtype = pd.CategoricalDtype(categories=cats)
    train_data[col] = train_data[col].astype(dtype)
    test_data[col] = test_data[col].astype(dtype)

In [56]:
attr = "wrong_call"

In [ ]:
train_features, test_features = train_data[feature_cols].drop(columns=["pitcher","batter"]) , test_data[feature_cols].drop(columns=["pitcher","batter"])
train_response, test_response = train_data[response_cols][attr].astype(np.int64).to_numpy() , test_data[response_cols][attr].astype(np.int64).to_numpy()


In [58]:
class MLP(nn.Module):
    def __init__(self, hidden_sizes=(50, 20)):
        super().__init__()
        layers = []
        for i, h in enumerate(hidden_sizes):
            layers.append(nn.LazyLinear(h) if i == 0 else nn.Linear(hidden_sizes[i - 1], h))
            layers.append(nn.ReLU())
        layers.append(nn.LazyLinear(2))   # binary classification -> 2 logits
        self.net = nn.Sequential(*layers)

    def forward(self, X):
        if not isinstance(X, torch.Tensor):
            X = torch.as_tensor(X)
        return self.net(X.float())

In [59]:
cat_cols = train_features.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = train_features.select_dtypes(include=["number"]).columns.tolist()

In [60]:
preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", RobustScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

In [ ]:
device="cuda"

mlp_model = NeuralNetClassifier(
    module=MLP,
    module__hidden_sizes=(50, 20),
    max_epochs=200,
    lr=1e-3,
    batch_size=1024,
    optimizer=torch.optim.Adam,
    optimizer__weight_decay=1e-3, 
    train_split=ValidSplit(0.1, stratified=True, random_state=6740),
    callbacks=[EarlyStopping(patience=5, monitor="valid_loss")],
    device=device,
    verbose=0,
    iterator_train__pin_memory=(device == "cuda"),
    iterator_train__num_workers=0,
    n_jobs=4,
)

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("mlp",  mlp_model),
])

param_grid = {
    "mlp__module__hidden_sizes":      [(50, 20), (100, 50)],
    "mlp__optimizer__weight_decay":   [1e-3, 1e-1],
    "mlp__batch_size":                [256, 512],
}
cv = TimeSeriesSplit(n_splits=5)

search = HalvingGridSearchCV(
    pipe, param_grid,
    cv=TimeSeriesSplit(n_splits=5),
    scoring="average_precision",
    factor=3, min_resources="exhaust",
    n_jobs=4,                          
)


search.fit(train_features, train_response)

print("Best params:", search.best_params_)
print("Best CV PR-AUC:", search.best_score_)

ValueError: 
All the 40 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
40 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\mmcdo\miniconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\mmcdo\miniconda3\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\mmcdo\miniconda3\Lib\site-packages\sklearn\pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "c:\Users\mmcdo\miniconda3\Lib\site-packages\skorch\classifier.py", line 170, in fit
    return super(NeuralNetClassifier,